# TickingLLM Experiment Runner

## Setup Env

In [ ]:
# === Environment Setup ===
# Install uv if not available, then sync dependencies
import subprocess, sys

# Install dependencies from pyproject.toml using uv
subprocess.run(["uv", "sync", "--extra", "notebook"], check=True)

# Ensure the notebook uses the .venv Python kernel
assert ".venv" in sys.executable or "uv" in sys.executable, \
    "Please select the .venv kernel: Kernel -> Change Kernel -> Python (.venv)"

print(f"Python: {sys.executable}")
print("Environment ready!")

## Download Models

Download model from huggingface. Replace YOUR_HF_TOKEN with your HuggingFace access token.

If you want to try different models, modify the model path in timelyllm/config.py

In [ ]:
# === Download Model ===
import os
from huggingface_hub import snapshot_download

MODEL_PATH = os.path.join(os.path.abspath('.'), 'model', 'Meta-Llama-3-8B-Instruct')
snapshot_download(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    local_dir=MODEL_PATH,
    token="YOUR_HF_TOKEN",  # Replace with your Hugging Face token
)

## Run before exp

In [ ]:
import subprocess
import os
import sys
import glob
import json
import time
import math
import statistics
import numpy as np
import matplotlib.pyplot as plt

os.environ['MKL_THREADING_LAYER'] = 'GNU'

PROJECT_ROOT = os.path.abspath('.')
TIMELYLLM_DIR = os.path.join(PROJECT_ROOT, 'timelyllm')
LOGS_DIR = os.path.join(TIMELYLLM_DIR, 'logs')
MODEL_PATH = os.path.join(PROJECT_ROOT, 'model', 'Meta-Llama-3-8B-Instruct')

sys.path.insert(0, TIMELYLLM_DIR)

RUN_DURATION = 10000

def get_log_path(preset):
    """Return the fixed log file path for a preset (named by experiment)."""
    return os.path.join(LOGS_DIR, f'{preset}.log')


def get_data_path(preset):
    """Return the fixed data JSON path for a preset."""
    return os.path.join(LOGS_DIR, f'{preset}_data.json')


def run_experiment(preset, run_duration=None, extra_args=None):
    """Run rtllm.py with a preset and --log-name. Returns the generated _data.json path."""
    cmd = [sys.executable, 'rtllm.py', '--preset', preset, '--log-name', preset,
           '--model-path', MODEL_PATH]
    if run_duration is not None:
        cmd += ['--run-duration', str(run_duration)]
    if extra_args:
        cmd += extra_args
    
    print(f"\n{'='*60}")
    print(f"Running preset: {preset}")
    print(f"Command: {' '.join(cmd)}")
    print(f"{'='*60}")
    
    proc = subprocess.run(cmd, cwd=TIMELYLLM_DIR, capture_output=False)
    if proc.returncode != 0:
        print(f"ERROR: preset {preset} exited with code {proc.returncode}")
        return None
    
    # Process the log with resdata
    log_file = get_log_path(preset)
    if not os.path.exists(log_file):
        print(f"ERROR: Log file not found: {log_file}")
        return None
    
    from resdata import read_log_file
    read_log_file(log_file)
    
    data_file = get_data_path(preset)
    print(f"Results saved to: {data_file}")
    return data_file

print("Helper function ready!")

## Experiment 7.4.1 — Performance Comparison with vLLM

Compare TimelyLLM with the vLLM baseline. <span style="color:green">**(~15 min)**</span>

In [ ]:
# Run 2 experiments for 7.4.1
for preset in ['exp741_vllm_high', 'exp741_timelyllm_high']:
    run_experiment(preset, run_duration=RUN_DURATION)

print("\n7.4.1 experiments complete. Log files:")
for preset in ['exp741_vllm_high', 'exp741_timelyllm_high']:
    print(f"  {preset}: {get_data_path(preset)}")

When the above cell finishes, the following files will appear under `timelyllm/logs/`:
- `exp741_vllm_high.log` / `exp741_vllm_high_data.json`
- `exp741_timelyllm_high.log` / `exp741_timelyllm_high_data.json`

Simply run the code in the next cell to directly plot the figures.

In [ ]:
# === 7.4.1 Plotting — uses fig_plot/performance_compare_vllm.py ===

%run fig_plot/performance_compare_vllm.py

## Experiment 7.4.3 — Scheduling Algorithms Comparison

Compare our algorithm against EDF and FCFS. <span style="color:green">**(~20 min)**</span>

In [ ]:
# Run 3 experiments for 7.4.3
for preset in ['exp743_fcfs', 'exp743_edf', 'exp743_timelyllm']:
    run_experiment(preset, run_duration=RUN_DURATION)

print("\n7.4.3 experiments complete. Log files:")
for preset in ['exp743_fcfs', 'exp743_edf', 'exp743_timelyllm']:
    print(f"  {preset}: {get_data_path(preset)}")

When the above cell finishes, the following files will appear under `timelyllm/logs/`:
- `exp743_fcfs.log` / `exp743_fcfs_data.json`
- `exp743_edf.log` / `exp743_edf_data.json`
- `exp743_timelyllm.log` / `exp743_timelyllm_data.json`

Simply run the code in the next cell to directly plot the figures.

In [ ]:
# === 7.4.3 Plotting — uses fig_plot/ablation_study_sched.py ===
%run fig_plot/ablation_study_sched.py

## Experiment 7.4.4 — Different Robotic Systems

Evaluate TimelyLLM on FLTRNN (robot arm), extending prior tests from TypeFly (drone). <span style="color:green">**(~25 min)**</span>

In [ ]:
# Run 2 experiments for 7.4.4 (FLTRNN)
for preset in ['exp744_fltrnn_vllm', 'exp744_fltrnn_timelyllm']:
    run_experiment(preset, run_duration=RUN_DURATION)

print("\n7.4.4 experiments complete. Log files:")
for preset in ['exp744_fltrnn_vllm', 'exp744_fltrnn_timelyllm']:
    print(f"  {preset}: {get_data_path(preset)}")

When the above cell finishes, the following files will appear under `timelyllm/logs/`:
- `exp744_fltrnn_vllm.log` / `exp744_fltrnn_vllm_data.json`
- `exp744_fltrnn_timelyllm.log` / `exp744_fltrnn_timelyllm_data.json`

Simply run the code in the next cell to directly plot the figures.

In [ ]:
# === 7.4.4 Plotting — uses robot_arm_res_fltrnn.py ===
%run fig_plot/robot_arm_res_fltrnn.py